Make sure to upload **clean** comments file.

1. Setup

In [1]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from tqdm import tqdm

!python -m spacy download en_core_web_sm

nlp = spacy.load("en_core_web_sm")
tqdm.pandas(desc="Progress")

df = pd.read_csv("Veritasium_youtube_comments_clean.csv")
print(df.shape)
df.head()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
(12899, 7)


,comment_id,author,updated_at,like_count,text,text_normalized,tokens_str
0,0,@DanielRenardAnimation,2019-09-19T18:28:21Z,5751,*Russian Cosmonaut spins a wingnut in space:* ...,russian cosmonaut spins a wingnut in space tel...,russian cosmonaut spins wingnut space tell one
1,1,@NLTops,2020-03-07T07:17:39Z,5244,Someone should tell the Flat Earthers of the D...,someone should tell the flat earthers of the d...,someone tell flat earthers dzhanibekov effect ...
2,2,@aviatordude1961,2021-07-26T02:06:48Z,4005,I thought the reason the Russians kept this a ...,i thought the reason the russians kept this a ...,thought reason russians kept secret going fema...
3,3,@alvirahesc7436,2021-04-10T12:21:42Z,3842,"""Babe, come over, im home alone""\n\n""No, babe,...","babe, come over, im home alone no, babe, im so...",babe come im home alone babe im solvin centuri...
4,4,@zeekjones1,2019-09-20T05:46:15Z,2814,Over 300 people broke their phone after watchi...,over 300 people broke their phone after watchi...,300 people broke phone watching video


2. Named Entity extraction (using spaCy)

In [2]:
def extract_entities(text):
    doc = nlp(str(text))
    return [(ent.text, ent.label_) for ent in doc.ents]

def extract_entity_texts(text):
    doc = nlp(str(text))
    return [ent.text for ent in doc.ents]

def extract_entity_types(text):
    doc = nlp(str(text))
    return [ent.label_ for ent in doc.ents]

df['entities'] = df['text_normalized'].progress_apply(extract_entities)
df['entity_texts'] = df['text_normalized'].progress_apply(extract_entity_texts)
df['entity_types'] = df['text_normalized'].progress_apply(extract_entity_types)

df[['text_normalized','entities']].head(10)

Progress: 100%|██████████| 12899/12899 [02:39<00:00, 80.88it/s] 


,text_normalized,entities
0,russian cosmonaut spins a wingnut in space tel...,"[(russian, NORP)]"
1,someone should tell the flat earthers of the d...,[]
2,i thought the reason the russians kept this a ...,"[(russians, NORP)]"
3,"babe, come over, im home alone no, babe, im so...","[(centuries, DATE)]"
4,over 300 people broke their phone after watchi...,"[(over 300, CARDINAL)]"
5,oh! so thats why my liquid filled bullets keep...,[]
6,this explaination is beautiful when you're act...,[]
7,750 australians didnt like the fact earth wont...,"[(750, CARDINAL)]"
8,video contains the phrase prove feynman wrong ...,[]
9,"as a kid, i would frequently watch my dad flip...","[(today, DATE)]"


In [3]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

# standard stopwords
stop_words = set(stopwords.words('english'))

# add contraction artifacts as custom stopwords (like Lab 1 showed)
custom_stopwords = {
    "im", "ll", "thats", "dont", "cant", "wont", "didnt", "ive",
    "youre", "theyre", "isnt", "wasnt", "wouldnt", "shouldnt",
    "id", "wed", "ve", "re", "s", "t", "n"
}

stop_words.update(custom_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
def clean_tokens(token_str):
    # Ensure the input is a string before splitting, handling potential NaN values
    if isinstance(token_str, float) and pd.isna(token_str):
        return '' # Return an empty string for NaN values
    tokens = str(token_str).split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [t for t in tokens if len(t) > 2]  # remove 1-2 char tokens too
    return ' '.join(tokens)

df['tokens_str'] = df['tokens_str'].apply(clean_tokens)

3. Keyword Extraction (TF-IDF)

In [6]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['tokens_str'].fillna(''))
feature_names = tfidf.get_feature_names_out()

def get_top_keywords(row_index, top_n=5):
    row = tfidf_matrix[row_index]
    scores = zip(feature_names, row.toarray()[0])
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return [word for word, score in sorted_scores if score > 0][:top_n]

df['keywords'] = [get_top_keywords(i) for i in range(len(df))]

df[['text_normalized','keywords']].head(10)

,text_normalized,keywords
0,russian cosmonaut spins a wingnut in space tel...,"[cosmonaut, russian, tell, wingnut, spins]"
1,someone should tell the flat earthers of the d...,"[ll, sooner, world flip, flat earthers, earthers]"
2,i thought the reason the russians kept this a ...,"[win, would always, gold, russians kept, gymna..."
3,"babe, come over, im home alone no, babe, im so...","[old math, home, centuries, alone, old]"
4,over 300 people broke their phone after watchi...,"[300, phone watching, broke, watching video, w..."
5,oh! so thats why my liquid filled bullets keep...,"[bullets, tumbling, liquid filled, filled, keep]"
6,this explaination is beautiful when you're act...,"[know thanks, thanks veritasium, explaination,..."
7,750 australians didnt like the fact earth wont...,"[australians, fact earth, fact, earth flip, like]"
8,video contains the phrase prove feynman wrong ...,"[phrase, clickbait, contains, video, prove fey..."
9,"as a kid, i would frequently watch my dad flip...","[imparting, would probably, inevitable, dad, a..."


4. Saving NER/Key CSV

In [7]:
df['entities_str'] = df['entities'].apply(lambda x: str(x))
df['entity_texts_str'] = df['entity_texts'].apply(lambda x: ', '.join(x))
df['entity_types_str'] = df['entity_types'].apply(lambda x: ', '.join(x))
df['keywords_str'] = df['keywords'].apply(lambda x: ', '.join(x))

output_cols = ['comment_id','author','updated_at','like_count',
               'text','text_normalized','tokens_str',
               'entities_str','entity_texts_str','entity_types_str',
               'keywords_str']

df[output_cols].to_csv("Veritasium_comments_NER-KEY_2.csv", index=False)
print(f"Saved {len(df)} enriched comments")

Saved 12899 enriched comments
